# EveryQuery — pooling probe + history-permutation experiments (T1.1 + T1.2)

This notebook runs two GPU experiments against the production checkpoint at
`gs://every-query-runs/outputs/2026-04-02/23-43-54/best_model.ckpt`:

1. **T1.1 — pooling linear probe.** Freeze the encoder, capture `last_hidden_state` on
   held-out tasks, and fit a logistic-regression probe per pooling strategy
   (`token_0`, `mean_history`, `max_history`, `concat_q_mean`). Compares pools on
   `occurs_auroc` per `(duration, code_bucket)`.
2. **T1.2 — history-permutation test.** Re-run inference with the patient-history
   token positions shuffled (query@0 and duration@1 left in place) and measure the
   per-code AUROC drop. A small drop = encoder is not using positional/temporal
   structure in the history.

Runs end-to-end on a Colab T4 (free tier) in ~25 minutes.
The retrospective evidence behind these experiments is in
[the priority list discussion](https://github.com/payalchandak/EveryQuery/issues/103).


## 1. Setup

You need a Colab GPU runtime and a paste of your GCS application-default credentials.


In [ ]:
# Install EveryQuery + analysis deps. ~3 minutes on Colab.
!pip -q install --upgrade pip
!pip -q install "git+https://github.com/payalchandak/EveryQuery.git@main" \
                google-cloud-storage polars scipy scikit-learn pyarrow nbformat


In [ ]:
# GCS auth.  Paste your application_default_credentials.json contents into the string below.
import json, os, pathlib
from google.cloud import storage

GCP_CREDS_JSON = '''
PASTE_YOUR_application_default_credentials_json_HERE
'''.strip()

cred_path = pathlib.Path.home() / '.config/gcloud/application_default_credentials.json'
cred_path.parent.mkdir(parents=True, exist_ok=True)
cred_path.write_text(GCP_CREDS_JSON)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(cred_path)

client = storage.Client(project='everyquery')
sample = list(client.list_blobs('every-query-runs', prefix='outputs/2026-04-02/23-43-54/', max_results=3))
print('GCS auth OK; sample blobs:')
for b in sample:
    print(' ', b.name, b.size)


## 2. Constants and code list

The production held-out evaluation uses 40 specific codes across 5 durations, split
into ID and OOD code buckets. We mirror that exactly so AUROCs are comparable to
the bucket's archived `eval_aucs_held_out_*.parquet`.


In [ ]:
RUN_DATE = '2026-04-02'
RUN_HASH = '23-43-54'
CKPT_BLOB = f'outputs/{RUN_DATE}/{RUN_HASH}/best_model.ckpt'
TASK_HASH = '7573f855c4b050a9d79d57fefd8a139c'
COHORT_PREFIX = 'meds/MIMIC_MEDS/MEDS_cohort/processed/'
EVAL_TASKS_PREFIX = f'eval_tasks/{TASK_HASH}/held_out/'
DURATIONS = [30, 90, 180, 365, 731]

# 40 production-eval codes × buckets, hard-coded from the bucket's eval AUC parquet so the
# notebook is reproducible without re-discovering them at runtime.
CODE_MAPPING = [
  {
    "code": "DIAGNOSIS//ICD//10//I25118",
    "code_slug": "DIAGNOSIS_ICD_10_I25118__ec52ca249f",
    "bucket": "ood"
  },
  {
    "code": "DIAGNOSIS//ICD//9//3320",
    "code_slug": "DIAGNOSIS_ICD_9_3320__2c078298ee",
    "bucket": "id"
  },
  {
    "code": "DIAGNOSIS//ICD//9//4271",
    "code_slug": "DIAGNOSIS_ICD_9_4271__67a74c8185",
    "bucket": "id"
  },
  {
    "code": "DIAGNOSIS//ICD//9//5856",
    "code_slug": "DIAGNOSIS_ICD_9_5856__25ad56ef20",
    "bucket": "id"
  },
  {
    "code": "DIAGNOSIS//ICD//9//7295",
    "code_slug": "DIAGNOSIS_ICD_9_7295__7a3f06dcb8",
    "bucket": "id"
  },
  {
    "code": "INFUSION_END//227536//value_[6.05,8.071587)",
    "code_slug": "INFUSION_END_227536_valu__e7fe2b52b0",
    "bucket": "ood"
  },
  {
    "code": "INFUSION_END//227536//value_[8.071587,9.379999)",
    "code_slug": "INFUSION_END_227536_valu__8e5b6bd753",
    "bucket": "id"
  },
  {
    "code": "INFUSION_END//229420//value_[10.687433,23.27545)",
    "code_slug": "INFUSION_END_229420_valu__f2fb7b22e3",
    "bucket": "id"
  },
  {
    "code": "INFUSION_START//220949",
    "code_slug": "INFUSION_START_220949__59ff06d700",
    "bucket": "ood"
  },
  {
    "code": "INFUSION_START//221794//value_[8.004926,10.000001)",
    "code_slug": "INFUSION_START_221794_va__d264c99562",
    "bucket": "id"
  },
  {
    "code": "INFUSION_START//225168//value_[284.02966,350.0)",
    "code_slug": "INFUSION_START_225168_va__9b30aa30f8",
    "bucket": "id"
  },
  {
    "code": "LAB//220224//mmHg//value_[89.0,98.0)",
    "code_slug": "LAB_220224_mmHg_value__8__cc297105aa",
    "bucket": "ood"
  },
  {
    "code": "LAB//220245//ml/min//value_[173.0,188.0)",
    "code_slug": "LAB_220245_ml_min_value___653ca4a391",
    "bucket": "id"
  },
  {
    "code": "LAB//220339//cmH2O//value_[10.0,12.0)",
    "code_slug": "LAB_220339_cmH2O_value____cd1dcba382",
    "bucket": "ood"
  },
  {
    "code": "LAB//224054//UNK//value_[2.0,3.0)",
    "code_slug": "LAB_224054_UNK_value__2.__1b2aadc9c1",
    "bucket": "ood"
  },
  {
    "code": "LAB//224665//UNK//value_[-inf,0.12)",
    "code_slug": "LAB_224665_UNK_value__-i__3130333121",
    "bucket": "ood"
  },
  {
    "code": "LAB//224690//insp/min//value_[14.0,16.0)",
    "code_slug": "LAB_224690_insp_min_valu__88dd3a0653",
    "bucket": "ood"
  },
  {
    "code": "LAB//225640//%//value_[0.3,0.6)",
    "code_slug": "LAB_225640_value__0.3_0.__989f723173",
    "bucket": "id"
  },
  {
    "code": "LAB//225672//IU/L//value_[30.0,42.0)",
    "code_slug": "LAB_225672_IU_L_value__3__8243ae2d61",
    "bucket": "id"
  },
  {
    "code": "LAB//226499//mL//value_[3150.0,inf)",
    "code_slug": "LAB_226499_mL_value__315__d66f1070ae",
    "bucket": "id"
  },
  {
    "code": "LAB//227073//mEq/L//value_[17.0,19.0)",
    "code_slug": "LAB_227073_mEq_L_value____7862b072d1",
    "bucket": "ood"
  },
  {
    "code": "LAB//227445//ng/mL//value_[11.0,20.0)",
    "code_slug": "LAB_227445_ng_mL_value____8991d8eeaf",
    "bucket": "id"
  },
  {
    "code": "LAB//228724//cm//value_[1.5,2.0)",
    "code_slug": "LAB_228724_cm_value__1.5__593ec20557",
    "bucket": "id"
  },
  {
    "code": "LAB//229663//cmH2O//value_[-inf,5.0)",
    "code_slug": "LAB_229663_cmH2O_value____acfb24ada8",
    "bucket": "ood"
  },
  {
    "code": "LAB//229694//UNK//value_[0.0,inf)",
    "code_slug": "LAB_229694_UNK_value__0.__c0569ca64d",
    "bucket": "ood"
  },
  {
    "code": "LAB//50957//mmol/L//value_[0.5,0.6)",
    "code_slug": "LAB_50957_mmol_L_value____5a81a2b385",
    "bucket": "ood"
  },
  {
    "code": "LAB//50964//mOsm/kg//value_[288.0,293.0)",
    "code_slug": "LAB_50964_mOsm_kg_value___47aa9d97dd",
    "bucket": "ood"
  },
  {
    "code": "LAB//51066//mg/24hr//value_[177.0,216.0)",
    "code_slug": "LAB_51066_mg_24hr_value___c409188bc2",
    "bucket": "ood"
  },
  {
    "code": "LAB//51274//sec//value_[14.1,15.3)",
    "code_slug": "LAB_51274_sec_value__14.__e28c7262eb",
    "bucket": "id"
  },
  {
    "code": "LAB//51769//UNK//value_[1.88,2.29)",
    "code_slug": "LAB_51769_UNK_value__1.8__77aba930f8",
    "bucket": "ood"
  },
  {
    "code": "MEDICATION//Carbidopa-Levodopa (25-100)//Administered",
    "code_slug": "MEDICATION_Carbidopa-Lev__5dfbebd91f",
    "bucket": "id"
  },
  {
    "code": "MEDICATION//Gabapentin//Delayed Administered",
    "code_slug": "MEDICATION_Gabapentin_De__0125170955",
    "bucket": "ood"
  },
  {
    "code": "MEDICATION//START//Mupirocin Nasal Ointment 2%",
    "code_slug": "MEDICATION_START_Mupiroc__c4bf6f14be",
    "bucket": "id"
  },
  {
    "code": "MEDICATION//STOP//Captopril",
    "code_slug": "MEDICATION_STOP_Captopri__b61170e7e2",
    "bucket": "ood"
  },
  {
    "code": "MEDICATION//STOP//Doxycycline Hyclate",
    "code_slug": "MEDICATION_STOP_Doxycycl__ea975c9b17",
    "bucket": "id"
  },
  {
    "code": "MEDS_DEATH",
    "code_slug": "MEDS_DEATH__d07ef7fed0",
    "bucket": "manual"
  },
  {
    "code": "PROCEDURE//ICD//9//7936",
    "code_slug": "PROCEDURE_ICD_9_7936__0044257467",
    "bucket": "id"
  },
  {
    "code": "SUBJECT_FLUID_OUTPUT//226600//mL//value_[20.0,30.0)",
    "code_slug": "SUBJECT_FLUID_OUTPUT_226__e8422b555f",
    "bucket": "ood"
  },
  {
    "code": "SUBJECT_FLUID_OUTPUT//226600//mL//value_[5.0,10.0)",
    "code_slug": "SUBJECT_FLUID_OUTPUT_226__fe4cba709f",
    "bucket": "id"
  },
  {
    "code": "SUBJECT_FLUID_OUTPUT//227510//mL//value_[25.0,50.0)",
    "code_slug": "SUBJECT_FLUID_OUTPUT_227__10903fbaa3",
    "bucket": "ood"
  }
]
SLUGS = [m['code_slug'] for m in CODE_MAPPING]
print(f'{len(CODE_MAPPING)} codes; ID={sum(1 for m in CODE_MAPPING if m["bucket"]=="id")}, '
      f'OOD={sum(1 for m in CODE_MAPPING if m["bucket"]=="ood")}')


## 3. Download artifacts

Pulls four things into `/content/eq/`:
- `best_model.ckpt` (1.79 GB)
- `MEDS_cohort/processed/data/held_out/*.nrt` + `metadata/codes.parquet` (~310 MB)
- the 40-code × 5-duration held-out task labels, **flattened** into one dir so
  `EveryQueryPytorchDataset` can read them with the shape it expects (~30 MB)


In [ ]:
import io, os, pathlib, polars as pl
from concurrent.futures import ThreadPoolExecutor
from google.cloud import storage

BUCKET = 'every-query-runs'
ROOT = pathlib.Path('/content/eq')
ROOT.mkdir(parents=True, exist_ok=True)
CKPT_LOCAL = ROOT / 'best_model.ckpt'
COHORT_LOCAL = ROOT / 'cohort'
TASKS_LOCAL = ROOT / 'tasks'
COHORT_LOCAL.mkdir(parents=True, exist_ok=True)
TASKS_LOCAL.mkdir(parents=True, exist_ok=True)

client = storage.Client(project='everyquery')
bucket = client.bucket(BUCKET)

def dl(blob_name, local):
    local = pathlib.Path(local)
    if local.exists() and local.stat().st_size > 0:
        return
    local.parent.mkdir(parents=True, exist_ok=True)
    bucket.blob(blob_name).download_to_filename(str(local))

# Checkpoint
print('Downloading checkpoint (1.79 GB)...')
dl(CKPT_BLOB, CKPT_LOCAL)
print('  done:', CKPT_LOCAL.stat().st_size, 'bytes')

# Tensorized cohort
print('Downloading cohort metadata + held_out shards...')
for blob in client.list_blobs(BUCKET, prefix=COHORT_PREFIX):
    rel = blob.name[len(COHORT_PREFIX):]
    if rel.startswith('data/held_out/') or rel.startswith('metadata/'):
        dl(blob.name, COHORT_LOCAL / rel)
print('  cohort dirs:', sorted(p.name for p in COHORT_LOCAL.iterdir()))

# Tasks: flatten <duration>/<code_slug>/<shard>.parquet -> single flat dir
# (write one parquet per (duration, code_slug) so shape matches what EQ_predict expects)
print('Downloading and flattening eval task labels...')
flat_dir = TASKS_LOCAL / 'held_out_flat'
flat_dir.mkdir(parents=True, exist_ok=True)

def flatten_one(slug, dur):
    prefix = f'{EVAL_TASKS_PREFIX}{dur}/{slug}/'
    parts = []
    for blob in client.list_blobs(BUCKET, prefix=prefix):
        if blob.name.endswith('.parquet'):
            parts.append(pl.read_parquet(io.BytesIO(blob.download_as_bytes())))
    if not parts:
        return None
    df = pl.concat(parts, how='vertical')
    out = flat_dir / f'{dur}_{slug}.parquet'
    df.write_parquet(out)
    return (dur, slug, df.height)

with ThreadPoolExecutor(max_workers=16) as ex:
    futs = []
    for slug in SLUGS:
        for dur in DURATIONS:
            futs.append(ex.submit(flatten_one, slug, dur))
    rows = sum((f.result() or (0,0,0))[2] for f in futs)
print(f'  total rows in flat task dir: {rows:,}')
print(f'  files: {len(list(flat_dir.iterdir()))}')


## 4. Build the held-out dataset and dataloader

We use `EveryQueryPytorchDataset` directly (no Lightning datamodule) so we can run a
plain forward loop and intercept `last_hidden_state` per batch.


In [ ]:
import torch
from meds_torchdata import MEDSPytorchDataset, MEDSTorchDataConfig
from meds_torchdata.types import BatchMode
from every_query.data.dataset import EveryQueryPytorchDataset

# Make EveryQueryPytorchDataset's task_labels_fps glob the flat dir.
# MEDSTorchDataConfig stores task_labels_dir; the underlying dataset reads everything
# matching `*.parquet` under it.
cfg = MEDSTorchDataConfig(
    tensorized_cohort_dir=str(COHORT_LOCAL),
    task_labels_dir=str(flat_dir),
    static_inclusion_mode='omit',
    seq_sampling_strategy='to_end',
    max_seq_len=256,
    batch_mode=BatchMode.SM,
)

ds = EveryQueryPytorchDataset(cfg, split='held_out')
print(f'held_out dataset: {len(ds):,} rows')

dl = torch.utils.data.DataLoader(
    ds, batch_size=32, shuffle=False, num_workers=2,
    collate_fn=ds.collate, pin_memory=True,
)
sample = next(iter(dl))
print(f'sample batch: code={tuple(sample.code.shape)}, '
      f'duration_days={tuple(sample.duration_days.shape) if sample.duration_days is not None else None}, '
      f'occurs={tuple(sample.occurs.shape)}, query={tuple(sample.query.shape)}')


## 5. Load the checkpoint

Loads with the same hyperparameters as the original training run (22 layers, hidden 768).
Frozen for inference; we'll only fit linear probes on top.


In [ ]:
from every_query.model.lightning_module import EveryQueryLightningModule

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

lm = EveryQueryLightningModule.load_from_checkpoint(str(CKPT_LOCAL))
lm.eval()
lm.to(device)
for p in lm.parameters():
    p.requires_grad_(False)

m = lm.model           # EveryQueryModel
hf = m.HF_model        # ModernBertModel
print(f'hidden_size={hf.config.hidden_size}, layers={hf.config.num_hidden_layers}, '
      f'max_position={hf.config.max_position_embeddings}')


## 6. Capture pass — extract `last_hidden_state` and four pooled embeddings per row

Forward pass over the full held-out dataloader. For each row we save:
- the **four pooled embeddings** (`token_0`, `mean_history`, `max_history`,
  `concat_q_mean`) computed from `last_hidden_state`
- ground-truth `occurs`, `censor`, `query`, `duration_days`
- the dataset row index so we can join back to `code` and `bucket`

Pooling definitions:
- `token_0` — the current production pool (`last_hidden_state[:, 0, :]`)
- `mean_history` — mean of positions 2..end excluding pad (skips query@0 and duration@1)
- `max_history` — max over positions 2..end excluding pad
- `concat_q_mean` — `concat(token_0, mean_history)` so the probe can learn its own weighting


In [ ]:
import numpy as np
from tqdm.auto import tqdm

def pool_embeddings(last_hidden, attn_mask):
    '''last_hidden: (B, L, H), attn_mask: (B, L) bool.  Returns dict of (B, H) tensors.'''
    B, L, H = last_hidden.shape
    # Position 0 = query token.  Position 1 = duration token (when duration_days is not None).
    # Treat positions >=2 as 'history'.
    history_mask = attn_mask.clone()
    history_mask[:, 0] = False
    history_mask[:, 1] = False  # duration token; safe to skip — when duration is None this is just position 1 which still belongs to the model anyway

    tok0 = last_hidden[:, 0, :]
    # mean_history: zero-out non-history positions, sum, divide by count
    h_mask = history_mask.unsqueeze(-1).to(last_hidden.dtype)
    h_count = h_mask.sum(dim=1).clamp(min=1)
    mean_h = (last_hidden * h_mask).sum(dim=1) / h_count
    # max_history: replace non-history with -inf then max
    masked = last_hidden.masked_fill(~history_mask.unsqueeze(-1), float('-inf'))
    max_h = masked.max(dim=1).values
    # Some rows (very short history) may be all -inf — replace with token_0 fallback
    bad = ~torch.isfinite(max_h).any(dim=-1)
    if bad.any():
        max_h[bad] = tok0[bad]
    concat = torch.cat([tok0, mean_h], dim=-1)
    return {'token_0': tok0, 'mean_history': mean_h, 'max_history': max_h, 'concat_q_mean': concat}

# Capture
pools_buf = {k: [] for k in ['token_0','mean_history','max_history','concat_q_mean']}
occ_buf, cens_buf, qry_buf, dur_buf, idx_buf = [], [], [], [], []
row_offset = 0

with torch.inference_mode():
    for batch in tqdm(dl, total=len(dl), desc='capture'):
        # Move to GPU
        for k, v in vars(batch).items():
            if isinstance(v, torch.Tensor):
                setattr(batch, k, v.to(device, non_blocking=True))
        # Build encoder inputs the same way EveryQueryModel does
        inputs = m._hf_inputs(batch)
        out = hf(**inputs)
        last_hidden = out.last_hidden_state
        attn_mask = inputs.get('attention_mask').bool()
        pools = pool_embeddings(last_hidden, attn_mask)
        for k, v in pools.items():
            pools_buf[k].append(v.cpu().to(torch.float32).numpy())
        occ_buf.append(batch.occurs.cpu().numpy())
        cens_buf.append(batch.censor.cpu().numpy())
        qry_buf.append(batch.query.cpu().numpy())
        dur_buf.append(batch.duration_days.cpu().numpy())
        idx_buf.append(np.arange(row_offset, row_offset + batch.code.shape[0]))
        row_offset += batch.code.shape[0]

pools_arr = {k: np.concatenate(v, axis=0) for k, v in pools_buf.items()}
occurs = np.concatenate(occ_buf, axis=0).astype(np.int64)
censor = np.concatenate(cens_buf, axis=0).astype(bool)
query = np.concatenate(qry_buf, axis=0).astype(np.int64)
duration = np.concatenate(dur_buf, axis=0).astype(np.float32)
row_index = np.concatenate(idx_buf, axis=0)
print({k: v.shape for k, v in pools_arr.items()})
print(f'occurs ones: {int(occurs.sum())}, censor=True: {int(censor.sum())}, total: {len(occurs)}')


## 7. Build per-row metadata for sub-population AUROC

The dataset's `schema_df` carries `code`, `bucket` (we'll join via the slug mapping), and
`duration_days`. We use it to label each captured row.


In [ ]:
# Per-row metadata from the dataset's schema_df.  schema_df is in dataloader (subject_id, prediction_time, ...) order,
# i.e. the same order our capture buffer sees because shuffle=False.
schema = ds.schema_df.collect() if hasattr(ds.schema_df, 'collect') else ds.schema_df
schema_pd = schema.to_pandas()
assert len(schema_pd) == len(occurs), (len(schema_pd), len(occurs))

slug_to_bucket = {m['code_slug']: m['bucket'] for m in CODE_MAPPING}
slug_to_code = {m['code_slug']: m['code'] for m in CODE_MAPPING}
code_to_bucket = {m['code']: m['bucket'] for m in CODE_MAPPING}
schema_pd['bucket'] = schema_pd['query'].map(code_to_bucket).fillna('unknown')
print(schema_pd['bucket'].value_counts())
print('per-duration counts:')
print(schema_pd['duration_days'].value_counts().sort_index())


## 8. Linear-probe AUROC per pool × duration × bucket

For each pool we fit `LogisticRegression` on probe-train (50% of rows, stratified by
duration) and evaluate AUROC on probe-eval (the other 50%). We report:
- overall `occurs_auroc` per pool
- `occurs_auroc` by duration
- `occurs_auroc` by code-bucket (id / ood)
- delta vs `token_0` (the production pool)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 50/50 split, stratified by duration_days
np.random.seed(140799)
durs = schema_pd['duration_days'].astype(int).to_numpy()
idx_train, idx_eval = train_test_split(
    np.arange(len(occurs)), test_size=0.5, random_state=140799, stratify=durs
)

# We evaluate occurs_auroc on the not-censored subset only (same convention as production eval)
not_cens = ~censor

def auroc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return float('nan')
    return float(roc_auc_score(y_true, y_score))

results = []
# NB: we use `mask` (not `m`) for boolean slice masks to avoid shadowing the loaded model `m = lm.model`.
for pool_name in ['token_0', 'mean_history', 'max_history', 'concat_q_mean']:
    X = pools_arr[pool_name]
    Xtr, Xev = X[idx_train], X[idx_eval]
    ytr_occ = occurs[idx_train]
    yev_occ = occurs[idx_eval]
    ncens_tr = not_cens[idx_train]
    ncens_ev = not_cens[idx_eval]
    fit_mask = ncens_tr
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, C=1.0)
    clf.fit(Xtr[fit_mask], ytr_occ[fit_mask])
    proba = clf.predict_proba(Xev)[:, 1]
    overall = auroc(yev_occ[ncens_ev], proba[ncens_ev])
    results.append({'pool': pool_name, 'slice': 'overall', 'n': int(ncens_ev.sum()), 'auroc': overall})
    for d in DURATIONS:
        mask = ncens_ev & (durs[idx_eval] == d)
        if mask.sum() < 30: continue
        results.append({'pool': pool_name, 'slice': f'd={d}', 'n': int(mask.sum()),
                        'auroc': auroc(yev_occ[mask], proba[mask])})
    bk = schema_pd['bucket'].to_numpy()[idx_eval]
    for b in ('id','ood'):
        mask = ncens_ev & (bk == b)
        if mask.sum() < 30: continue
        results.append({'pool': pool_name, 'slice': f'bucket={b}', 'n': int(mask.sum()),
                        'auroc': auroc(yev_occ[mask], proba[mask])})

import pandas as pd
res_df = pd.DataFrame(results)
print(res_df.pivot(index='slice', columns='pool', values='auroc').round(4))

# Delta vs token_0
piv = res_df.pivot(index='slice', columns='pool', values='auroc')
delta = piv.subtract(piv['token_0'], axis=0).round(4)
print('\nDelta AUROC (pool - token_0):')
print(delta)


## 9. T1.2 — history-permutation test

For each batch we shuffle the order of positions 2..end (the patient history) while
leaving position 0 (query) and position 1 (duration) fixed. We then re-run the **trained
heads** (no probe) on the shuffled sequence and compute AUROC.

A small AUROC drop = the encoder ignores the order of history events (i.e. it is
behaving like a bag-of-codes model with a duration scalar). A large drop = ordering
matters and a time-aware encoder will likely matter even more.

To keep this fast we run on **one duration (180d)** and a random subset of subjects.


In [ ]:
# Subset to duration=180 for speed
mask_180 = (durs == 180) & not_cens
sub_idx = np.where(mask_180)[0]
np.random.seed(140799)
np.random.shuffle(sub_idx)
sub_idx = sub_idx[:8000]  # ~8k examples is enough for stable per-bucket AUROC

def fetch_row(i):
    return ds[i]

def collate_one(items):
    return ds.collate(items)

orig_probs, perm_probs, orig_y, codes, buckets = [], [], [], [], []

CHUNK = 32
with torch.inference_mode():
    for chunk_start in tqdm(range(0, len(sub_idx), CHUNK), desc='permutation'):
        idxs = sub_idx[chunk_start:chunk_start+CHUNK]
        items = [fetch_row(int(i)) for i in idxs]
        batch = collate_one(items)
        for k, v in vars(batch).items():
            if isinstance(v, torch.Tensor):
                setattr(batch, k, v.to(device, non_blocking=True))
        # Original prediction
        _, out_orig = m(batch)
        orig_probs.append(out_orig.occurs_probs.detach().cpu().numpy().reshape(-1))
        # Permuted: shuffle history positions in batch.code per row.
        # NB: batch.code has shape (B, L) with [query@0, history@1..end, pad..].  The duration
        # token only exists inside inputs_embeds (injected by m._hf_inputs), NOT in batch.code,
        # so the history range to shuffle in batch.code is positions 1..valid_end-1.
        code_orig = batch.code.clone()
        pad = (code_orig == batch.PAD_INDEX)
        for r in range(code_orig.shape[0]):
            valid_end = int((~pad[r]).sum().item())
            if valid_end > 2:  # need query + >=2 history events
                perm = torch.randperm(valid_end - 1, device=device) + 1
                code_orig[r, 1:valid_end] = batch.code[r, perm]
        # numeric_value / numeric_value_mask / time_delta_days are unused by the model
        # (verified in src/every_query/model/model.py:_hf_inputs), so leaving them un-permuted is fine.
        batch.code = code_orig
        _, out_perm = m(batch)
        perm_probs.append(out_perm.occurs_probs.detach().cpu().numpy().reshape(-1))
        orig_y.append(batch.occurs.cpu().numpy().reshape(-1))
        codes.extend([schema_pd['query'].iloc[int(i)] for i in idxs])
        buckets.extend([schema_pd['bucket'].iloc[int(i)] for i in idxs])

orig_probs = np.concatenate(orig_probs); perm_probs = np.concatenate(perm_probs); orig_y = np.concatenate(orig_y)

print(f'overall AUROC orig:    {auroc(orig_y, orig_probs):.4f}  (n={len(orig_y)})')
print(f'overall AUROC permuted:{auroc(orig_y, perm_probs):.4f}')

# Per-code AUROC delta (use m_c for the per-code mask to avoid shadowing the model `m`)
codes_arr = np.array(codes); buckets_arr = np.array(buckets)
per_code_delta = []
for c_ in sorted(set(codes_arr)):
    m_c = codes_arr == c_
    if m_c.sum() < 50: continue
    a_o = auroc(orig_y[m_c], orig_probs[m_c])
    a_p = auroc(orig_y[m_c], perm_probs[m_c])
    per_code_delta.append({'code': c_, 'bucket': buckets_arr[m_c][0], 'n': int(m_c.sum()),
                          'auroc_orig': round(a_o,4), 'auroc_perm': round(a_p,4),
                          'drop': round(a_o - a_p, 4)})

import pandas as pd
perm_df = pd.DataFrame(per_code_delta).sort_values('drop', ascending=False)
print('\nPer-code AUROC drop from history shuffle (sorted, larger = more position-sensitive):')
print(perm_df.to_string(index=False))
print('\nMean AUROC drop overall:', perm_df['drop'].mean().round(4))
print('Mean AUROC drop ID codes:', perm_df.loc[perm_df['bucket']=='id','drop'].mean().round(4))
print('Mean AUROC drop OOD codes:', perm_df.loc[perm_df['bucket']=='ood','drop'].mean().round(4))


## 10. Save results

Writes the pooling probe table and the permutation table back to the bucket so the
priority-list write-up can cite these numbers.


In [ ]:
import io, polars as pl

OUT_PREFIX = 'analysis/pooling_permutation/'

def upload(df, name):
    buf = io.BytesIO()
    pl.from_pandas(df).write_parquet(buf)
    bucket.blob(OUT_PREFIX + name).upload_from_string(buf.getvalue())
    print('uploaded gs://every-query-runs/'+OUT_PREFIX+name)

upload(res_df, 'pooling_probe_aucs.parquet')
upload(perm_df, 'permutation_per_code_aucs.parquet')
print('Done.')
